In [54]:
!pip install python-dotenv


Defaulting to user installation because normal site-packages is not writeable


In [55]:

import dotenv
import os

dotenv.load_dotenv(dotenv_path='.env')



True

In [56]:
import base64
import json
import os
import time
from pathlib import Path

import requests


class IMGBBFile:
    def __init__(self, filename, name, mime, extension, url):
        self.filename = filename
        self.name = name
        self.mime = mime
        self.extension = extension
        self.url = url


class IMGBBData:
    def __init__(self, id, title, url_viewer, url, display_url, width, height, size, time, expiration, image, thumb, medium, delete_url):
        self.id = id
        self.title = title
        self.url_viewer = url_viewer
        self.url = url
        self.display_url = display_url
        self.width = width
        self.height = height
        self.size = size
        self.time = time
        self.expiration = expiration
        self.image = image
        self.thumb = thumb
        self.medium = medium
        self.delete_url = delete_url


class IMGBBResponse:
    def __init__(self, data, success, status):
        self.data = data
        self.success = success
        self.status = status


cache = {}


def upload_image_to_imgbb(image_base64, name, api_key):
    if cache.get(name):
        return cache.get(name)

    
    url = "https://api.imgbb.com/1/upload"
    payload = {
        "key": api_key,
        "image": image_base64,
        "name": name,
    }

    try:
        response = requests.post(url, data=payload, timeout=60)
        response_data = response.json()

        if not response_data.get("success"):
            print(f"Error: Image upload failed with status code {response.status_code} and message: {response_data.get('error', {}).get('message', 'No error message')}")
            return {
                "result": False,
                "error": "Image upload failed",
                "data": IMGBBResponse(response_data.get("data"), False, response.status_code),
            }

        result = {
            "result": True,
            "error": "",
            "data": IMGBBResponse(response_data.get("data"), True, response.status_code),
        }
        cache[name] = result
        return result

    except Exception as e:
        print(f"Error: {str(e)}")
        return {
            "result": False,
            "error": str(e),
            "data": IMGBBResponse(None, False, 503),
        }



In [57]:

workspace_root = Path(r"d:/AndroidStudioProjects/PhotoMakerAssets")
manifest_path = workspace_root / "background_manifest.json"


In [58]:
# progress bar
import sys
from ipywidgets import IntProgress, Text
from IPython.display import display

from IPython.display import clear_output



In [59]:
def find_local_image_path(category_name, image_id):
    category_dir = workspace_root / "background" / category_name
    if not category_dir.exists():
        return None

    for ext in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
        candidate = category_dir / f"{image_id}{ext}"
        if candidate.exists():
            return candidate

    for candidate in sorted(category_dir.glob(f"{image_id}*")):
        if candidate.is_file():
            return candidate

    return None


def write_manifest(manifest):
    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
        handle.write("\n")


def get_thumb_url(response_data):
    thumb = response_data.get("thumb")
    if isinstance(thumb, dict):
        return thumb.get("url") or None
    return thumb or None


def upload_manifest_images(manifest, delay_seconds=1):
    uploaded_count = 0
    
    total_count = sum(len(category.get("backgrounds", [])) for category in manifest.get("categories", []))
    global_count = IntProgress(min=0, max=total_count)
    f = IntProgress(min=0, max=10) # instantiate the bar
    text = Text(value='Hello World!', disabled=True)
    global_count_text = Text(value=f'Total items to process: {total_count}', disabled=True)
    display(text)
    display(f)
    display(global_count)
    for category in manifest.get("categories", []):
        f = IntProgress(min=0, max=len(category.get("backgrounds", []))) # instantiate the bar
        text = Text(value=f'Processing category: {category.get("name")}', disabled=True)
        display(global_count)
        display(global_count_text)
        display(text)
        display(f)
        
        
        
        category_name = category.get("name")
        for background in category.get("backgrounds", []):
            if background.get("imgbb"):
                background.pop("imgbb_delete_url", None)
                background.pop("imgbb_error", None)
                text.value = f"Skip {category_name}/{background.get('id')}: already processed"
                global_count.value += 1
                global_count_text.value = f'Total items to process: {total_count}, Processed: {global_count.value}'
                continue

            image_id = background.get("id")
            if not image_id:
                global_count.value += 1
                continue

            image_path = find_local_image_path(category_name, image_id)
            if not image_path:
                text.value = f"Skip {category_name}/{image_id}: local file not found"
                global_count.value += 1
                global_count_text.value = f'Total items to process: {total_count}, Processed: {global_count.value}'
                continue

            image_bytes = image_path.read_bytes()
            image_base64 = base64.b64encode(image_bytes).decode("utf-8")
            retry_count = 0
            api_keys = [
                os.environ.get("IMGBB_API", "").strip(),
                os.environ.get("IMGBB_API_2", "").strip(),
            ]
            while retry_count < 3:
                result = upload_image_to_imgbb(image_base64, image_path.name, api_keys[retry_count % len(api_keys)])
                if result.get("result"):
                    response_obj = result.get("data")
                    response_data = getattr(response_obj, "data", None) or {}
                    background.pop("imgbb_delete_url", None)
                    background["imgbb"] = response_data.get("url")
                    background["imgbb_display_url"] = response_data.get("display_url")
                    background["imgbb_thumb"] = get_thumb_url(response_data)
                    text.value = f"Uploaded {category_name}/{image_id} -> {background['imgbb']}"
                    background.pop("imgbb_error", None)
                    break
                else:
                    background.pop("imgbb_delete_url", None)
                    background.pop("imgbb_error", None)
                    text.value = f"Failed {category_name}/{image_id}: {result.get('error')}"
                    time.sleep(120)  # wait for 2 minutes before continuing to avoid rate limiting
                    retry_count += 1

            f.value += 1
            global_count.value += 1
            global_count_text.value = f'Total items to process: {total_count}, Processed: {global_count.value}'
            text.value = f'Processed {uploaded_count + 1} items'

            write_manifest(manifest)
            uploaded_count += 1
            time.sleep(delay_seconds)

        # clear display of progress bar and text
        clear_output(wait=True)

    print(f"Finished processing {uploaded_count} items")
    return manifest


In [ ]:

with manifest_path.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)
    upload_manifest_images(manifest)
    


Text(value='Hello World!', disabled=True)

IntProgress(value=0, max=10)

IntProgress(value=0, max=1348)

IntProgress(value=0, max=1348)

Text(value='Total items to process: 1348', disabled=True)

Text(value='Processing category: Abstract', disabled=True)

IntProgress(value=0, max=109)